# 20 — Bài báo, bước 3b: mô hình tổng chi phí và kịch bản triển khai (máy local)

**Phạm vi**

Tổng chi phí trong 12 tháng của 28 cấu hình đã chấm ở notebook 17, theo khối lượng review, chi phí một nhãn, giá phần cứng và chế độ
triển khai; khối lượng hòa vốn giữa cấu hình có huấn luyện và cấu hình gọi API; cấu hình rẻ nhất đạt từng ngưỡng Macro-F1. Thiết kế
pre-register ở amendment v3.8.

**Đo và giả định**

Số nhãn, thời gian train, thời gian suy luận và USD API là số đo của bước 2. Chi phí một nhãn, giá giờ máy, khối lượng và chính sách
kiểm tra trôi dữ liệu là tham số giả định trong `03_configs/frozen/paper_cost_model.yaml`, dùng để phân tích độ nhạy; kết luận dựa trên
vùng hòa vốn và thứ tự chi phí, không dựa trên con số tuyệt đối.

In [1]:
import os, sys, json, datetime
import numpy as np, pandas as pd, yaml
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification import cost_model as CM
pd.set_option('display.width', 230); pd.set_option('display.max_columns', 30); pd.set_option('display.max_rows', 100)
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
txt = open(os.path.join(ROOT, '03_configs', 'frozen', 'paper_cost_model.yaml'), encoding='utf-8').read()
assert txt.startswith('# FROZEN') and 'amendment v3.8' in txt
CFG = yaml.safe_load(txt)['cost_model']
COMP = CM.components(pd.read_csv(os.path.join(RES, 'paper_step2_costs.csv')), pd.read_csv(os.path.join(RES, 'paper_step2_pareto.csv')),
                     CFG['labels_initial'])
print('Số lần kiểm tra trôi dữ liệu trong', CFG['horizon_months'], 'tháng:', CM.drift_checks(CFG))
show = COMP.copy()
show['USD API / 1.000 review'] = show.pop('USD API / review') * 1000
show['giây suy luận / 1.000 review'] = show.pop('giây suy luận / review') * 1000
print(show.sort_values('Macro-F1', ascending=False).round(3).to_string(index=False))

Số lần kiểm tra trôi dữ liệu trong 12 tháng: 1
                         cấu hình      mã kiến trúc  Macro-F1  gọi API  có huấn luyện phần cứng  nhãn ban đầu  lần chạy dò lưới  giây train / lần chạy  USD API / 1.000 review  giây suy luận / 1.000 review
             Cascade LLM few-shot     L→L   cascade    81.289     True          False     không           367                 0                  0.000                   0.206                      1103.455
               Phẳng LLM few-shot L phẳng     phẳng    79.987     True          False     không           367                 0                  0.000                   0.266                       791.483
           LLM few-shot → PhoBERT     L→P    hybrid    74.611     True           True       GPU          1550                 9                 10.688                   0.173                       913.504
          LLM few-shot → ViSoBERT     L→V    hybrid    73.285     True           True       GPU          1550                 9      

## Tổng chi phí 12 tháng ở kịch bản gốc

**Kịch bản gốc**

Chi phí một nhãn 0,10 USD, GPU 0,35 USD/giờ; bảng cho sáu cấu hình đại diện ở bốn mức khối lượng và hai chế độ triển khai.

In [2]:
NAMES = ['Phẳng TF-IDF', 'Phẳng ViSoBERT', 'Phẳng PhoBERT', 'LLM few-shot → PhoBERT', 'Phẳng LLM few-shot', 'Cascade LLM few-shot']
B = CFG['base_scenario']
base = pd.concat([CM.tco(COMP, CFG, v, B['label_cost_usd'], B['gpu_usd_per_hour'], m) for m in CFG['deployment'] for v in CFG['monthly_volume']])
pv = base[base['cấu hình'].isin(NAMES)].pivot_table(index=['cấu hình', 'Macro-F1'], columns=['triển khai', 'review / tháng'], values='tổng USD')
print('Tổng USD trong 12 tháng (chi phí một nhãn 0,10 USD; GPU 0,35 USD/giờ):'); print(pv.loc[NAMES].round(1).to_string())

Tổng USD trong 12 tháng (chi phí một nhãn 0,10 USD; GPU 0,35 USD/giờ):
triển khai                         batch                         endpoint                        
review / tháng                   1000    10000   100000  1000000  1000    10000   100000  1000000
cấu hình               Macro-F1                                                                  
Phẳng TF-IDF           62.752909   220.0   220.0   220.0   220.0    658.0   658.0   658.0   658.0
Phẳng ViSoBERT         66.700642   220.0   220.1   220.2   221.7   3286.0  3286.0  3286.0  3286.0
Phẳng PhoBERT          70.818585   220.0   220.1   220.3   222.4   3286.0  3286.0  3286.0  3286.0
LLM few-shot → PhoBERT 74.610587   223.3   251.6   534.6  3365.1   3288.2  3306.9  3494.1  5365.3
Phẳng LLM few-shot     79.987210    75.1   103.8   391.2  3265.0     75.1   103.8   391.2  3265.0
Cascade LLM few-shot   81.288627    74.3    96.6   319.5  2548.8     74.3    96.6   319.5  2548.8


## Khối lượng hòa vốn

**Cách đọc**

Khối lượng review mỗi tháng mà từ đó cấu hình có huấn luyện (chi phí cố định lớn) rẻ hơn cấu hình gọi API (chi phí tăng theo số review),
theo chi phí một nhãn. "không bao giờ" nghĩa là cấu hình có huấn luyện không rẻ hơn ở bất kỳ khối lượng nào; 0 nghĩa là luôn rẻ hơn.

In [3]:
A = ['Phẳng TF-IDF', 'Phẳng ViSoBERT', 'Phẳng PhoBERT', 'Cascade PhoBERT']
BB = ['Cascade LLM few-shot', 'Phẳng LLM few-shot', 'LLM few-shot → PhoBERT']
rows = []
for m in CFG['deployment']:
    for g in CFG['gpu_usd_per_hour']:
        for a in A:
            for b in BB:
                for lc in CFG['label_cost_usd']:
                    rows.append({'triển khai': m, 'giá GPU / giờ': g, 'có huấn luyện': a, 'gọi API': b, 'chi phí một nhãn': lc,
                                 'hòa vốn (review / tháng)': CM.break_even(COMP, CFG, a, b, lc, g, m)})
BE = pd.DataFrame(rows)
fmt = lambda x: 'không bao giờ' if np.isinf(x) else f'{x:,.0f}'.replace(',', '.')
for m in CFG['deployment']:
    t = BE[(BE['triển khai'] == m) & (BE['giá GPU / giờ'] == B['gpu_usd_per_hour'])].pivot_table(
        index=['có huấn luyện', 'gọi API'], columns='chi phí một nhãn', values='hòa vốn (review / tháng)', aggfunc='first')
    print(f'Hòa vốn — {m}, GPU {B["gpu_usd_per_hour"]} USD/giờ (review / tháng):'); print(t.map(fmt).to_string()); print()

Hòa vốn — batch, GPU 0.35 USD/giờ (review / tháng):
chi phí một nhãn                          0.02    0.05    0.10     0.20     0.50
có huấn luyện   gọi API                                                         
Cascade PhoBERT Cascade LLM few-shot    11.949  29.930  59.899  119.836  299.649
                LLM few-shot → PhoBERT       0       0       0        0        0
                Phẳng LLM few-shot       9.254  23.199  46.441   92.926  232.378
Phẳng PhoBERT   Cascade LLM few-shot    11.946  29.925  59.889  119.817  299.601
                LLM few-shot → PhoBERT       0       0       0        0        0
                Phẳng LLM few-shot       9.252  23.196  46.435   92.914  232.349
Phẳng TF-IDF    Cascade LLM few-shot    11.916  29.877  59.813  119.685  299.299
                LLM few-shot → PhoBERT       0       0       0        0        0
                Phẳng LLM few-shot       9.231  23.164  46.386   92.831  232.164
Phẳng ViSoBERT  Cascade LLM few-shot    11.941  29.914  5

## Cấu hình rẻ nhất theo ngưỡng chất lượng

**Cách đọc**

Với mỗi kịch bản, cấu hình có tổng chi phí 12 tháng thấp nhất trong các cấu hình có Macro-F1 Hold-out đạt ngưỡng; chi phí chênh dưới 1%
hoặc dưới 10 USD trong 12 tháng được coi như bằng nhau và khi đó cấu hình có Macro-F1 cao hơn được chọn. Phần "không gọi API" áp ràng buộc không được gửi review ra dịch vụ
bên ngoài (quyền riêng tư hoặc chính sách dữ liệu); với ràng buộc này không cấu hình nào đạt ngưỡng 80.

In [4]:
SC = CM.scenarios(COMP, CFG)
DEC = []
for q in CFG['quality_thresholds']:
    for allow in (True, False):
        d = CM.cheapest(SC, q, allow)
        d.insert(0, 'được gọi API', allow); d.insert(0, 'ngưỡng Macro-F1', q)
        DEC.append(d)
DEC = pd.concat([d for d in DEC if len(d)], ignore_index=True)
for allow in (True, False):
    t = DEC[(DEC['được gọi API'] == allow) & (DEC['giá GPU / giờ'] == B['gpu_usd_per_hour']) & DEC['chi phí một nhãn'].isin([0.02, 0.10, 0.50])]
    if t.empty:
        continue
    pvt = t.pivot_table(index=['triển khai', 'review / tháng'], columns=['ngưỡng Macro-F1', 'chi phí một nhãn'], values='mã', aggfunc='first')
    print(f"Cấu hình rẻ nhất đạt ngưỡng — {'được' if allow else 'không được'} gọi API, GPU {B['gpu_usd_per_hour']} USD/giờ:")
    print(pvt.fillna('—').to_string()); print()

Cấu hình rẻ nhất đạt ngưỡng — được gọi API, GPU 0.35 USD/giờ:
ngưỡng Macro-F1                 60                         70                     80          
chi phí một nhãn              0.02     0.10     0.50     0.02     0.10     0.50 0.02 0.10 0.50
triển khai review / tháng                                                                     
batch      1000                L→L      L→L      L→L      L→L      L→L      L→L  L→L  L→L  L→L
           10000               L→L      L→L      L→L      L→L      L→L      L→L  L→L  L→L  L→L
           100000          P phẳng  P phẳng      L→L  P phẳng  P phẳng      L→L  L→L  L→L  L→L
           1000000         P phẳng  P phẳng  P phẳng  P phẳng  P phẳng  P phẳng  L→L  L→L  L→L
endpoint   1000                L→L      L→L      L→L      L→L      L→L      L→L  L→L  L→L  L→L
           10000               L→L      L→L      L→L      L→L      L→L      L→L  L→L  L→L  L→L
           100000              L→L      L→L      L→L      L→L      L→L      L→L  L→

## Hình tổng chi phí theo khối lượng

**Nội dung**

Kịch bản gốc (chi phí một nhãn 0,10 USD, GPU 0,35 USD/giờ), hai panel cho batch và endpoint, trục log cả hai chiều. ViSoBERT và PhoBERT
phẳng có chi phí gần như trùng nhau (cùng GPU, cùng số nhãn) nên vẽ chung một đường; TF-IDF vẽ nét đứt vì ở chế độ batch đường của nó
trùng với hai mô hình kia.

In [5]:
VOL = np.unique(np.round(np.logspace(3, 6.3, 40)).astype(int))
FIG = ['Phẳng TF-IDF', 'Phẳng PhoBERT', 'LLM few-shot → PhoBERT', 'Phẳng LLM few-shot', 'Cascade LLM few-shot']
CUR = CM.tco_curves(COMP, CFG, FIG, B['label_cost_usd'], B['gpu_usd_per_hour'], VOL)
TXT = {'vi': {'x': 'Review cần phân loại mỗi tháng (thang log)', 'y': 'Tổng chi phí 12 tháng (USD, thang log)',
              'modes': {'batch': 'Chạy theo lô (trả theo giờ máy dùng)', 'endpoint': 'Máy chủ bật liên tục (730 giờ/tháng)'}},
       'en': {'x': 'Reviews classified per month (log scale)', 'y': 'Total cost over 12 months (USD, log scale)',
              'modes': {'batch': 'Batch (pay per machine hour used)', 'endpoint': 'Always-on endpoint (730 h/month)'}}}
LAB = {'vi': {**{n: n for n in FIG}, 'Phẳng PhoBERT': 'Phẳng ViSoBERT / PhoBERT'},
       'en': {'Phẳng TF-IDF': 'Flat TF-IDF', 'Phẳng PhoBERT': 'Flat ViSoBERT / PhoBERT',
              'LLM few-shot → PhoBERT': 'LLM → PhoBERT (hybrid)', 'Phẳng LLM few-shot': 'Flat LLM', 'Cascade LLM few-shot': 'Cascade LLM'}}
for lang in TXT:
    CM.tco_figure(CUR, FIG, LAB[lang], os.path.join(RES, f'paper_cost_tco_{lang}.png'), TXT[lang], styles={'Phẳng TF-IDF': '--'})
print('Hình: paper_cost_tco_vi.png, paper_cost_tco_en.png')

Hình: paper_cost_tco_vi.png, paper_cost_tco_en.png


## Lưu kết quả

**Các file**

Thành phần chi phí, mọi kịch bản, khối lượng hòa vốn và bảng cấu hình rẻ nhất, tiền tố `paper_cost_` trong `05_experiments/rq1_classification/`.

In [6]:
COMP.to_csv(os.path.join(RES, 'paper_cost_components.csv'), index=False, encoding='utf-8-sig')
SC.to_csv(os.path.join(RES, 'paper_cost_scenarios.csv'), index=False, encoding='utf-8-sig')
BE.to_csv(os.path.join(RES, 'paper_cost_break_even.csv'), index=False, encoding='utf-8-sig')
DEC.to_csv(os.path.join(RES, 'paper_cost_decision.csv'), index=False, encoding='utf-8-sig')
json.dump({'run_date': datetime.date.today().strftime('%d/%m/%Y'), 'amendment': 'v3.8', 'drift_checks': CM.drift_checks(CFG)},
          open(os.path.join(RES, 'paper_cost_run_info.json'), 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith('paper_cost_')))

Đã lưu: ['paper_cost_break_even.csv', 'paper_cost_components.csv', 'paper_cost_decision.csv', 'paper_cost_run_info.json', 'paper_cost_scenarios.csv', 'paper_cost_tco_en.png', 'paper_cost_tco_vi.png']


## Nhận xét

**Cấu trúc chi phí**

Với cấu hình có huấn luyện, chi phí gần như chỉ là chi phí nhãn: 2.200 nhãn trong 12 tháng (1.550 ban đầu và 650 cho một lần kiểm tra trôi
dữ liệu), tức 220 USD ở mức 0,10 USD một nhãn; train và suy luận theo lô chỉ thêm dưới 3 USD mỗi năm kể cả ở 1 triệu review mỗi tháng.
Cấu hình chỉ dùng LLM cần ít nhãn hơn (717 nhãn, 71,7 USD) nhưng trả API tăng tuyến tính theo khối lượng: 0,206 USD cho 1.000 review với
cascade và 0,266 USD với bản phẳng.

**Khối lượng hòa vốn**

Ở chế độ batch, mô hình phẳng có huấn luyện rẻ hơn cascade LLM từ khoảng 12.000 review mỗi tháng (chi phí một nhãn 0,02 USD) đến khoảng
300.000 (0,50 USD), và khoảng 60.000 ở kịch bản gốc 0,10 USD. Nếu phải bật máy chủ GPU liên tục, điểm hòa vốn lên khoảng 1,25–1,54 triệu
review mỗi tháng; TF-IDF chạy CPU hòa vốn ở khoảng 190.000–480.000. Hybrid LLM → PhoBERT luôn đắt hơn mô hình phẳng có huấn luyện vì phải
trả cả nhãn lẫn API, nên chỉ đáng dùng khi cần thêm chất lượng (74,6 so với 70,8).

**Lựa chọn theo điều kiện**

Muốn Macro-F1 từ 80 trở lên thì chỉ có LLM, và cascade LLM là bản rẻ nhất. Ở ngưỡng 70, cascade LLM rẻ nhất khi dưới khoảng 10.000 review
mỗi tháng hoặc khi phải bật máy chủ liên tục; PhoBERT phẳng rẻ nhất khi chạy theo lô từ khoảng 100.000 review mỗi tháng trở lên, trừ khi
chi phí nhãn cao. Khi không được gửi review ra API bên ngoài, PhoBERT phẳng là lựa chọn cho ngưỡng 70, còn TF-IDF trên CPU là lựa chọn rẻ
cho ngưỡng 60 khi phải chạy liên tục; không cấu hình nào đạt 80 với ràng buộc này.

**Giới hạn**

Chi phí một nhãn, giá giờ máy và chính sách kiểm tra trôi dữ liệu là giả định; giờ gán nhãn không được ghi nhận (v3.0). Chất lượng của mỗi
cấu hình lấy từ Hold-out Shopee và có thể khác trên ứng dụng khác, như kết quả Lazada đã cho thấy với LLM. Giá API là giá của snapshot đã
khóa tại thời điểm chạy và có thể thay đổi.